# Расчёт метрик обращений NYC 311

ЛР № 5, расчёт метрик на витрине `service_requests` и подготовка данных для ЛР № 7

> Как различались число обращений и медианное время до закрытия по типам проблем и ответственным агентствам в Манхэттене в январе 2025 года?

Подробные определения метрик, пояснение оконной функции, итоги проверок и вывод находятся в [docs/metrics.md](../docs/metrics.md).
Промежуточные файлы не сохраняются, результаты остаются во временных таблицах DuckDB и переменных Python для продолжения в ЛР № 7

In [1]:
from pathlib import Path

DATASET_SLUG = "nyc311_manhattan"
SOURCE_INGESTED_ON = "2026-10-14"
MART_PATH = (
    f"s3://mart/{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}/service_requests.parquet"
)
PERIOD_START = "2025-01-01"
PERIOD_END = "2025-02-01"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Вход {MART_PATH}")
print(f"Период по created_date [{PERIOD_START}, {PERIOD_END})")

Вход s3://mart/nyc311_manhattan/source_ingested_on=2026-10-14/service_requests.parquet
Период по created_date [2025-01-01, 2025-02-01)


## Подключение и входная выборка

Настройки S3 читаются из локального `.env`, витрина один раз загружается в память DuckDB во временную таблицу `analysis_base` с фильтром по периоду января 2025 года

In [2]:
import os
from html import escape

import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(PROJECT_ROOT / ".env")
con = duckdb.connect(":memory:")
con.execute("INSTALL httpfs FROM 'https://extensions.duckdb.org'")
con.execute("LOAD httpfs")
settings = {
    "s3_endpoint": os.environ["S3_ENDPOINT"],
    "s3_access_key_id": os.environ["S3_ACCESS_KEY"],
    "s3_secret_access_key": os.environ["S3_SECRET_KEY"],
    "s3_region": "us-east-1",
    "s3_url_style": "path",
    "s3_use_ssl": False,
}
for name, value in settings.items():
    con.execute(f"SET {name} = ?", [value])


def sql_literal(value):
    return "'" + value.replace("'", "''") + "'"


def show_rows(columns, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def show_query(sql, params=None):
    result = con.execute(sql, params or [])
    rows = result.fetchall()
    show_rows([column[0] for column in result.description], rows)
    return rows

In [3]:
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW service_requests_mart AS
    SELECT *
    FROM read_parquet({sql_literal(MART_PATH)}, hive_partitioning = false)
""")
required_fields = {
    "unique_key",
    "created_date",
    "closed_date",
    "complaint_type",
    "agency",
    "request_count",
    "resolution_hours",
    "quality_missing_complaint_type",
    "quality_missing_agency",
}
mart_schema = {row[0]: row[1] for row in con.execute("DESCRIBE service_requests_mart").fetchall()}
assert required_fields.issubset(mart_schema)

con.execute(f"""
    CREATE OR REPLACE TEMP TABLE analysis_base AS
    SELECT *
    FROM service_requests_mart
    WHERE created_date >= TIMESTAMP {sql_literal(PERIOD_START)}
      AND created_date < TIMESTAMP {sql_literal(PERIOD_END)}
""")
input_summary = show_query("""
    SELECT
        count(*) AS rows_count,
        count(resolution_hours) AS closed_count,
        count(*) FILTER (WHERE closed_date IS NULL) AS unclosed_count,
        count(*) FILTER (WHERE quality_missing_complaint_type) AS missing_type_flags,
        count(*) FILTER (WHERE quality_missing_agency) AS missing_agency_flags,
        min(created_date) AS first_created,
        max(created_date) AS last_created
    FROM analysis_base
""")[0]
print(f"Полей в витрине {len(mart_schema)}")

rows_count,closed_count,unclosed_count,missing_type_flags,missing_agency_flags,first_created,last_created
61304,60268,1036,0,0,2025-01-01 00:00:37,2025-01-31 23:59:48


Полей в витрине 9


## Общий итог

Три метрики считаются по всем допустимым обращениям января 2025 года.
Записи без даты закрытия участвуют в числе обращений и доле незакрытых, но исключаются из медианы времени до закрытия

In [4]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE overall_metrics AS
    SELECT
        sum(request_count) AS requests_count,
        count(resolution_hours) AS closed_count,
        count(*) FILTER (WHERE closed_date IS NULL) AS unclosed_count,
        median(resolution_hours) AS median_resolution_hours,
        100.0 * count(*) FILTER (WHERE closed_date IS NULL)
            / nullif(sum(request_count), 0) AS unclosed_share_pct
    FROM analysis_base
""")
overall_rows = show_query("""
    SELECT
        requests_count,
        closed_count,
        unclosed_count,
        round(median_resolution_hours, 2) AS median_resolution_hours,
        round(unclosed_share_pct, 2) AS unclosed_share_pct
    FROM overall_metrics
""")
overall_raw = con.execute("SELECT * FROM overall_metrics").fetchone()
assert overall_raw is not None

requests_count,closed_count,unclosed_count,median_resolution_hours,unclosed_share_pct
61304,60268,1036,31.87,1.69


## Основная группировка по агентствам

Одна строка таблицы `agency_metrics` соответствует одному агентству за весь январь 2025 года.
Оконная функция рассчитывает долю каждого агентства в общем числе обращений с защитой от нулевого знаменателя через `nullif`

In [5]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE agency_metrics AS
    WITH grouped AS (
        SELECT
            agency,
            sum(request_count) AS requests_count,
            count(resolution_hours) AS closed_count,
            count(*) FILTER (WHERE closed_date IS NULL) AS unclosed_count,
            median(resolution_hours) AS median_resolution_hours,
            100.0 * count(*) FILTER (WHERE closed_date IS NULL)
                / nullif(sum(request_count), 0) AS unclosed_share_pct
        FROM analysis_base
        GROUP BY agency
    )
    SELECT
        *,
        100.0 * requests_count
            / nullif(sum(requests_count) OVER (), 0) AS share_of_requests_pct
    FROM grouped
""")
agency_rows = show_query("""
    SELECT
        agency,
        requests_count,
        closed_count,
        unclosed_count,
        round(median_resolution_hours, 2) AS median_resolution_hours,
        round(unclosed_share_pct, 2) AS unclosed_share_pct,
        round(share_of_requests_pct, 2) AS share_of_requests_pct
    FROM agency_metrics
    ORDER BY requests_count DESC, agency
""")
agency_raw = con.execute("""
    SELECT *
    FROM agency_metrics
    ORDER BY requests_count DESC, agency
""").fetchall()
print(f"Групп по агентствам {len(agency_raw)}")

agency,requests_count,closed_count,unclosed_count,median_resolution_hours,unclosed_share_pct,share_of_requests_pct
HPD,24333,24268,65,84.25,0.27,39.69
NYPD,17656,17656,0,0.91,0.0,28.8
DSNY,4642,4630,12,22.22,0.26,7.57
DEP,3239,3222,17,26.42,0.52,5.28
DOT,2812,2774,38,23.32,1.35,4.59
DOHMH,2037,1977,60,43.9,2.95,3.32
TLC,1724,1644,80,1578.74,4.64,2.81
DHS,1537,791,746,15.76,48.54,2.51
DOB,1511,1511,0,63.62,0.0,2.46
EDC,866,866,0,3403.37,0.0,1.41


Групп по агентствам 14


## Уточняющая группировка и выборка для распределения

В `agency_complaint_metrics` к агентству добавлен тип проблемы, отсутствующие сочетания не дописываются нулями.
В `resolution_distribution` выбраны отдельные заполненные длительности для анализа распределения и сравнения агентств в ЛР № 7

In [6]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE agency_complaint_metrics AS
    SELECT
        agency,
        complaint_type,
        sum(request_count) AS requests_count,
        count(resolution_hours) AS closed_count,
        count(*) FILTER (WHERE closed_date IS NULL) AS unclosed_count,
        median(resolution_hours) AS median_resolution_hours,
        100.0 * count(*) FILTER (WHERE closed_date IS NULL)
            / nullif(sum(request_count), 0) AS unclosed_share_pct
    FROM analysis_base
    GROUP BY agency, complaint_type
""")
agency_complaint_preview = show_query("""
    SELECT
        agency,
        complaint_type,
        requests_count,
        closed_count,
        unclosed_count,
        round(median_resolution_hours, 2) AS median_resolution_hours,
        round(unclosed_share_pct, 2) AS unclosed_share_pct
    FROM agency_complaint_metrics
    ORDER BY requests_count DESC, agency, complaint_type
    LIMIT 15
""")
agency_complaint_totals = show_query("""
    SELECT
        agency,
        count(*) AS complaint_types,
        sum(requests_count) AS requests_count,
        sum(closed_count) AS closed_count,
        count(*) FILTER (WHERE closed_count = 0) AS groups_without_closed
    FROM agency_complaint_metrics
    GROUP BY agency
    ORDER BY requests_count DESC, agency
""")
agency_complaint_rows = con.execute("""
    SELECT *
    FROM agency_complaint_metrics
    ORDER BY requests_count DESC, agency, complaint_type
""").fetchall()
print(f"Строк в уточняющей группировке {len(agency_complaint_rows)}")

con.execute("""
    CREATE OR REPLACE TEMP TABLE resolution_distribution AS
    SELECT
        agency,
        complaint_type,
        resolution_hours
    FROM analysis_base
    WHERE resolution_hours IS NOT NULL
""")
distribution_summary = show_query("""
    SELECT
        count(*) AS closed_count,
        count(*) FILTER (WHERE resolution_hours = 0) AS zero_hours_count,
        round(min(resolution_hours), 2) AS min_hours,
        round(quantile_cont(resolution_hours, 0.25), 2) AS p25_hours,
        round(median(resolution_hours), 2) AS median_hours,
        round(quantile_cont(resolution_hours, 0.75), 2) AS p75_hours,
        round(max(resolution_hours), 2) AS max_hours
    FROM resolution_distribution
""")[0]
resolution_values = [
    row[0]
    for row in con.execute("""
        SELECT resolution_hours
        FROM resolution_distribution
        ORDER BY resolution_hours
    """).fetchall()
]
print(f"Отдельных значений длительности {len(resolution_values)}")

agency,complaint_type,requests_count,closed_count,unclosed_count,median_resolution_hours,unclosed_share_pct
HPD,HEAT/HOT WATER,15538,15526,12,65.58,0.08
NYPD,Noise - Residential,5666,5666,0,1.05,0.0
NYPD,Illegal Parking,4903,4903,0,1.03,0.0
HPD,UNSANITARY CONDITION,1853,1845,8,719.95,0.43
NYPD,Noise - Commercial,1843,1843,0,0.62,0.0
DEP,Noise,1748,1747,1,48.17,0.06
HPD,PLUMBING,1698,1691,7,243.02,0.41
NYPD,Noise - Street/Sidewalk,1630,1630,0,0.77,0.0
NYPD,Encampment,1192,1192,0,0.78,0.0
HPD,PAINT/PLASTER,1147,1139,8,267.72,0.7


agency,complaint_types,requests_count,closed_count,groups_without_closed
HPD,13,24333,24268,0
NYPD,22,17656,17656,0
DSNY,22,4642,4630,1
DEP,10,3239,3222,0
DOT,17,2812,2774,0
DOHMH,25,2037,1977,3
TLC,8,1724,1644,3
DHS,2,1537,791,0
DOB,13,1511,1511,0
EDC,1,866,866,0


Строк в уточняющей группировке 146


closed_count,zero_hours_count,min_hours,p25_hours,median_hours,p75_hours,max_hours
60268,933,0.0,1.77,31.87,104.65,15276.07


Отдельных значений длительности 60268


## Проверка расчётов

Проверяются совпадение итогов по группировкам, сумма долей, число заполненных длительностей, объём выборки распределения, корректное объединение доли незакрытых и совпадение общей медианы

In [7]:
check_rows = show_query("""
    WITH checks AS (
        SELECT
            1 AS check_no,
            'Сумма обращений по агентствам равна общему итогу' AS check_name,
            (SELECT sum(requests_count) FROM agency_metrics)::DOUBLE AS actual,
            (SELECT requests_count FROM overall_metrics)::DOUBLE AS expected
        UNION ALL
        SELECT
            2,
            'Сумма обращений по агентствам и типам проблем равна общему итогу',
            (SELECT sum(requests_count) FROM agency_complaint_metrics)::DOUBLE,
            (SELECT requests_count FROM overall_metrics)::DOUBLE
        UNION ALL
        SELECT
            3,
            'Сумма долей агентств равна 100 %',
            (SELECT sum(share_of_requests_pct) FROM agency_metrics)::DOUBLE,
            100.0::DOUBLE
        UNION ALL
        SELECT
            4,
            'Число заполненных длительностей по группам совпадает с базой',
            (SELECT sum(closed_count) FROM agency_complaint_metrics)::DOUBLE,
            (SELECT count(resolution_hours) FROM analysis_base)::DOUBLE
        UNION ALL
        SELECT
            5,
            'Число строк в выборке распределения равно числу закрытых обращений',
            (SELECT count(*) FROM resolution_distribution)::DOUBLE,
            (SELECT closed_count FROM overall_metrics)::DOUBLE
        UNION ALL
        SELECT
            6,
            'Общая доля без даты закрытия совпадает по базе и по суммам групп',
            (SELECT 100.0 * sum(unclosed_count) / nullif(sum(requests_count), 0)
             FROM agency_metrics)::DOUBLE,
            (SELECT unclosed_share_pct FROM overall_metrics)::DOUBLE
        UNION ALL
        SELECT
            7,
            'Общая медиана по выборке распределения совпадает с общим итогом',
            (SELECT median(resolution_hours) FROM resolution_distribution)::DOUBLE,
            (SELECT median_resolution_hours FROM overall_metrics)::DOUBLE
    )
    SELECT
        check_name,
        round(actual, 4) AS actual,
        round(expected, 4) AS expected,
        abs(actual - expected) < 1e-6 AS passed
    FROM checks
    ORDER BY check_no
""")
assert all(row[3] for row in check_rows)
print(f"Проверок пройдено {len(check_rows)} из {len(check_rows)}")

check_name,actual,expected,passed
Сумма обращений по агентствам равна общему итогу,61304.0,61304.0,True
Сумма обращений по агентствам и типам проблем равна общему итогу,61304.0,61304.0,True
Сумма долей агентств равна 100 %,100.0,100.0,True
Число заполненных длительностей по группам совпадает с базой,60268.0,60268.0,True
Число строк в выборке распределения равно числу закрытых обращений,60268.0,60268.0,True
Общая доля без даты закрытия совпадает по базе и по суммам групп,1.6899,1.6899,True
Общая медиана по выборке распределения совпадает с общим итогом,31.8736,31.8736,True


Проверок пройдено 7 из 7


## Продолжение в ЛР № 7

Подключение DuckDB остаётся открытым, в памяти доступны временные таблицы `analysis_base`, `overall_metrics`, `agency_metrics`, `agency_complaint_metrics` и `resolution_distribution`, а также переменные Python `overall_raw`, `agency_raw`, `agency_complaint_rows`, `resolution_values` и `distribution_summary`.
Описание метрик, проверок и выводы собраны в [docs/metrics.md](../docs/metrics.md)